[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/23_Reasoning.ipynb)

# 推理模块 —— 实用指南

Semantica 的 `reasoning` 模块从已有的事实和知识图谱中推导出新知识。它在同一个门面(facade)之后提供了多种策略:

- **`Reasoner`** —— 统一的门面,支持前向链、后向链以及一次性的 `infer_facts`
- **`DatalogReasoner`** —— 半朴素 Datalog 不动点求值,支持带变量的查询
- **`ExplanationGenerator`** —— 为推断出的结论生成人类可读的解释和推理路径
- 另外还有较底层的引擎:`ReteEngine`、`SPARQLReasoner`、`GraphReasoner`、时态推理

本 Notebook 依次介绍门面、Datalog 引擎和解释。所有 API 均已对照 `semantica/reasoning/` 验证。

In [1]:
!pip install -q semantica

## 1)使用 `Reasoner` 门面进行前向链推理

事实是简单的 `Predicate(args)` 字符串。规则使用 `IF <条件> THEN <结论>`,变量采用 `?x` 风格。`forward_chain()` 会推导出所有可能的结论,并返回 `InferenceResult` 对象的列表。

In [2]:
from semantica.reasoning import Reasoner

reasoner = Reasoner()

reasoner.add_fact("Person(John)")
reasoner.add_fact("Person(Jane)")
reasoner.add_rule("IF Person(?x) THEN Human(?x)")

results = reasoner.forward_chain()
print(f"Inferred {len(results)} new facts")
for res in results:
    print(f"  {res.conclusion}  (rule: {res.rule_used.name}, confidence: {res.confidence})")

Status,Action,Module,Submodule,Progress,ETA,Rate,Time,Extracted
✅,Semantica is reasoning,🤔 reasoning,Reasoner,100.0%,-,-,0.00s,-
✅,Semantica is reasoning,🤔 reasoning,DatalogReasoner,100.0%,-,-,0.00s,-
✅,Semantica is reasoning,🤔 reasoning,ExplanationGenerator,100.0%,-,-,0.00s,-


🔄 Semantica is reasoning: Performing forward chaining 🤔 reasoning Reasoner |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

Inferred 2 new facts
  Human(Jane)  (rule: Rule 1, confidence: 1.0)
  Human(John)  (rule: Rule 1, confidence: 1.0)


## 2)使用 `infer_facts` 进行一次性推断

`infer_facts(facts, rules)` 会将给定的事实和规则**添加**到当前 `Reasoner` 实例中,运行前向链直至不动点,并以字符串形式返回推导出的事实。它不会重置实例的现有状态 —— 如果你需要在多次运行之间相互隔离,请先创建一个全新的 `Reasoner()`。

In [3]:
from semantica.reasoning import Reasoner

derived = Reasoner().infer_facts(
    facts=["WorksFor(John, Acme)", "WorksFor(Jane, Acme)"],
    rules=["IF WorksFor(?x, ?y) THEN Employee(?x, ?y)"],
)
derived

['Employee(Jane, Acme)', 'Employee(John, Acme)']

## 3)后向链:证明一个目标

`backward_chain(goal)` 从结论出发,沿规则向后推导。它返回证明了该目标的 `InferenceResult`,或者返回 `None`。

In [4]:
from semantica.reasoning import Reasoner

reasoner = Reasoner()
reasoner.add_fact("Person(John)")
reasoner.add_rule("IF Person(?x) THEN Human(?x)")

proof = reasoner.backward_chain("Human(John)")
print(proof.conclusion if proof else "not provable")
print("premises:", proof.premises if proof else None)

Human(John)
premises: ['Person(John)']


## 4)重复运行的安全性

`add_rule` 会对条件和结论相同的规则去重,因此重新执行设置单元格(Jupyter 中常见的重复运行)不会产生重复的规则 —— 参见 issue #732。

In [5]:
from semantica.reasoning import Reasoner

reasoner = Reasoner()
reasoner.add_fact("Person(John)")

# Simulate a Jupyter cell re-run: add the same rule twice
r1 = reasoner.add_rule("IF Person(?x) THEN Human(?x)")
r2 = reasoner.add_rule("IF Person(?x) THEN Human(?x)")

len(reasoner.rules)

Skipping duplicate rule (same conditions/conclusion as 'rule_1'): IF Person(?x) THEN Human(?x)


1

## 5)Datalog 推理

`DatalogReasoner` 使用经典的 Datalog 语法(`head :- body.`)和半朴素不动点求值。查询以字典列表的形式返回变量绑定 —— 使用大写变量来询问*哪些*事实成立。

In [6]:
from semantica.reasoning import DatalogReasoner

datalog = DatalogReasoner()
datalog.add_fact("parent(tom, mary)")
datalog.add_fact("parent(mary, ann)")
datalog.add_rule("grandparent(X, Z) :- parent(X, Y), parent(Y, Z)")

datalog.derive_all()
datalog.query("grandparent(X, Z)")

[{'X': 'tom', 'Z': 'ann'}]

## 6)为推断出的结论生成解释

`ExplanationGenerator` 会把 `InferenceResult` 对象转换成结构化的 `Explanation` 和 `ReasoningPath` 记录,使智能体能够展示它们*为什么*相信某个推导出的事实。

In [7]:
from semantica.reasoning import Reasoner, ExplanationGenerator

reasoner = Reasoner()
reasoner.add_fact("Person(John)")
reasoner.add_rule("IF Person(?x) THEN Human(?x)")
results = reasoner.forward_chain()

gen = ExplanationGenerator()
explanation = gen.generate_explanation(results[0])
path = gen.show_reasoning_path(results[0])

type(explanation).__name__, type(path).__name__

('Explanation', 'ReasoningPath')

## 总结

| 任务 | API |
|---|---|
| 推导所有新事实 | `Reasoner.forward_chain()` |
| 一次性推断 | `Reasoner.infer_facts(facts, rules)` |
| 证明一个目标 | `Reasoner.backward_chain(goal)` |
| Datalog 不动点 | `DatalogReasoner.derive_all()` + `query("p(X, Y)")` |
| 解释一个结论 | `ExplanationGenerator.generate_explanation(result)` |

另请参阅 `semantica/reasoning/reasoning_usage.md`,以及 `ReteEngine`、`SPARQLReasoner` 和时态推理的模块文档字符串。